# param-group-dict-list — faded example 2: complete the rank-based decay split

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `param-group-dict-list`. The last cell reports your progress on the `Config: param-group dict list` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Config: param-group dict list` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`param-group-dict-list`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "param-group-dict-list"
DD_SUBTOPIC = "Config: param-group dict list"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The no-decay group holds parameters with `p.dim() <= 1` (biases, norm scales); the decay group holds `p.dim() > 1` weight matrices. The partition uses a list comprehension over the parameters.

## Faded exercise 2

Complete the partition so `decay` holds rank>1 params and `no_decay` holds rank<=1 params. Fill in the decay-eligible comprehension.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(6)

def make_three_groups(encoder, head, encoder_lr, head_lr, weight_decay):
    enc = list(encoder.parameters())
    decay = None  # TODO: fill in this step — read the prompt cell above
    no_decay = [p for p in enc if p.dim() <= 1]
    return [
        {'params': decay, 'lr': encoder_lr, 'weight_decay': weight_decay},
        {'params': no_decay, 'lr': encoder_lr, 'weight_decay': 0.0},
        {'params': list(head.parameters()), 'lr': head_lr, 'weight_decay': weight_decay},
    ]

enc, head = nn.Linear(6, 6), nn.Linear(6, 3)
print([len(g['params']) for g in make_three_groups(enc, head, 1e-4, 1e-2, 0.01)])


def _test():
    enc, head = nn.Linear(6, 6), nn.Linear(6, 3)
    groups = make_three_groups(enc, head, 1e-4, 1e-2, 0.01)
    assert len(groups) == 3
    # encoder: 1 weight matrix (decay), 1 bias (no-decay)
    assert all(p.dim() > 1 for p in groups[0]['params'])
    assert all(p.dim() <= 1 for p in groups[1]['params'])
    assert groups[1]['weight_decay'] == 0.0
    # no param dropped or duplicated
    total = sum(len(g['params']) for g in groups)
    assert total == len(list(enc.parameters())) + len(list(head.parameters()))


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn

t.manual_seed(6)

def make_three_groups(encoder, head, encoder_lr, head_lr, weight_decay):
    enc = list(encoder.parameters())
    decay = [p for p in enc if p.dim() > 1]
    no_decay = [p for p in enc if p.dim() <= 1]
    return [
        {'params': decay, 'lr': encoder_lr, 'weight_decay': weight_decay},
        {'params': no_decay, 'lr': encoder_lr, 'weight_decay': 0.0},
        {'params': list(head.parameters()), 'lr': head_lr, 'weight_decay': weight_decay},
    ]

enc, head = nn.Linear(6, 6), nn.Linear(6, 3)
print([len(g['params']) for g in make_three_groups(enc, head, 1e-4, 1e-2, 0.01)])
```
</details>